# 08. 테스트 주문 구매 상품 예측 (Submission)

## 파이프라인
```
v4 데이터로 모델 학습
    ↓
orders.csv에서 eval_set=test 주문 추출 (75,000건)
    ↓
테스트 유저의 prior 구매 기록으로 후보(user, product) 쌍 생성
    ↓
피처 생성 (기존 prep 파일 + 구매시점 + 공동구매)
    ↓
모델 예측 → 최적 threshold 적용
    ↓
order_products 형태 CSV 저장
```

## 출력 파일
- `data/prep/submission_order_products.csv`
- 컬럼: `order_id`, `product_id`, `reorder_prob`
  (order_products__train.csv와 동일한 형태)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

_candidates = [
    Path(r'c:\Users\gksal\capstone-kpick\K-Pick'),
    Path.cwd(),
    Path.cwd().parent,
]
BASE = next((p for p in _candidates if (p / 'data' / 'prep').exists()), None)
if BASE is None:
    raise FileNotFoundError(f'data/prep 폴더를 찾을 수 없습니다. 현재 경로: {Path.cwd()}')

RAW  = BASE / 'data' / 'raw'
PREP = BASE / 'data' / 'prep'
print(f'BASE : {BASE}')
print('라이브러리 로드 완료')

---
## 1. 모델 학습 (v4 데이터 기반)
07_ml_model_v4에서 검증된 설정 그대로 사용

In [ ]:
print('v4 학습 데이터 로드 중...')
_v4 = PREP / 'k-pick_total_v4.csv'
if not _v4.exists():
    raise FileNotFoundError('k-pick_total_v4.csv 없음 — 06_feat_advanced.ipynb를 먼저 실행하세요.')

df_train = pd.read_csv(_v4)
print(f'v4 행: {len(df_train):,}  |  컬럼: {df_train.shape[1]}')

DROP_COLS = ['user_id', 'product_id', 'order_id', 'label']
X_all = df_train.drop(columns=[c for c in DROP_COLS if c in df_train.columns])
y_all = df_train['label']

# 결측치 처리
X_all = X_all.fillna(X_all.median(numeric_only=True))

# 학습용/검증용 분리 (최적 threshold 탐색용)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_all, y_all, test_size=0.15, random_state=42, stratify=y_all
)

scale_pos_weight = (y_tr == 0).sum() / (y_tr == 1).sum()
print(f'\nscale_pos_weight: {scale_pos_weight:.2f}')
print(f'Train: {len(X_tr):,}행  |  Val: {len(X_val):,}행')

In [ ]:
print('=== LightGBM 학습 중... ===')
model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=20,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.log_evaluation(period=100)
    ]
)

# validation 기반 최적 threshold 탐색
prob_val = model.predict_proba(X_val)[:, 1]
thresholds = np.arange(0.05, 0.55, 0.01)
best_thr = max(thresholds, key=lambda t: f1_score(y_val, (prob_val >= t).astype(int)))
best_f1  = f1_score(y_val, (prob_val >= best_thr).astype(int))

# 모델이 학습한 피처 목록 저장 (test 피처 정렬에 사용)
MODEL_FEATURES = model.feature_name_

print(f'\n최적 threshold : {best_thr:.2f}')
print(f'Val F1-Score   : {best_f1:.4f}')
print(f'학습 피처 수   : {len(MODEL_FEATURES)}')

---
## 2. 테스트 후보군 생성
eval_set=test 유저 × prior에서 구매한 상품 → 예측 대상 (user, product) 쌍

In [ ]:
print('원본 데이터 로드 중...')
orders = pd.read_csv(
    RAW / 'orders.csv',
    usecols=['order_id','user_id','eval_set','order_number',
             'order_dow','order_hour_of_day','days_since_prior_order']
)
prior_products = pd.read_csv(
    RAW / 'order_products__prior.csv',
    usecols=['order_id','product_id']
)

# test 주문 정보
test_orders = orders[orders['eval_set'] == 'test'][
    ['order_id','user_id','order_dow','order_hour_of_day','days_since_prior_order']
].copy()
test_orders['days_since_prior_order'] = test_orders['days_since_prior_order'].fillna(0)

# prior 주문에 user_id 붙이기
prior_order_info = orders[orders['eval_set'] == 'prior'][['order_id','user_id']]
prior_with_user  = prior_products.merge(prior_order_info, on='order_id', how='inner')

# 후보: test 유저 × prior에서 구매한 상품 (유니크)
test_user_ids = test_orders['user_id'].unique()
candidates = (
    prior_with_user[prior_with_user['user_id'].isin(test_user_ids)]
    [['user_id','product_id']]
    .drop_duplicates()
)

# test 주문 정보 병합
candidates = candidates.merge(
    test_orders[['user_id','order_id','order_dow','order_hour_of_day','days_since_prior_order']],
    on='user_id', how='left'
)

print(f'test 유저 수         : {len(test_user_ids):,}')
print(f'후보 (user, product) : {len(candidates):,}')
print(f'유저당 평균 후보 수  : {len(candidates)/len(test_user_ids):.1f}')
candidates.head()

---
## 3. 피처 생성
### 3-1. prep 파일 기반 피처 병합 (user-product / user / product)

In [ ]:
print('기존 피처 파일 병합 중...')

# user-product 피처
up_feat = pd.read_csv(PREP / 'user_product_features_final.csv')
candidates = candidates.merge(up_feat, on=['user_id','product_id'], how='left')
print(f'user_product_features 병합 후 컬럼: {candidates.shape[1]}')
del up_feat

# user 피처
u_feat = pd.read_csv(PREP / 'user_features.csv')
candidates = candidates.merge(u_feat, on='user_id', how='left')
print(f'user_features 병합 후 컬럼       : {candidates.shape[1]}')
del u_feat

# product 피처 (aisle_id, department_id 중복 제거)
p_feat = pd.read_csv(PREP / 'prod_features.csv')
p_feat = p_feat.drop(columns=['aisle_id','department_id'], errors='ignore')
candidates = candidates.merge(p_feat, on='product_id', how='left')
print(f'prod_features 병합 후 컬럼       : {candidates.shape[1]}')
del p_feat

# _x/_y 중복 컬럼 제거
dup = [c for c in candidates.columns if c.endswith(('_x','_y'))]
if dup:
    print(f'중복 컬럼 제거: {dup}')
    candidates.drop(columns=dup, inplace=True)

print(f'\n최종 컬럼 수: {candidates.shape[1]}')

### 3-2. 구매 시점 피처 생성 (test 유저 기준)

In [ ]:
print('구매 시점 피처 계산 중...')

# test 유저의 prior 주문 — 누적 날짜 계산
prior_orders_info = orders[orders['eval_set'] == 'prior'].copy()
prior_orders_info['days_since_prior_order'] = prior_orders_info['days_since_prior_order'].fillna(0)
prior_orders_sorted = prior_orders_info[
    prior_orders_info['user_id'].isin(test_user_ids)
].sort_values(['user_id','order_number'])

prior_orders_sorted['cum_days'] = (
    prior_orders_sorted.groupby('user_id')['days_since_prior_order'].cumsum()
)

# prior 구매 기록 + 누적 날짜 병합
pp_test = (
    prior_products
    .merge(prior_orders_sorted[['order_id','user_id','order_number','cum_days']],
           on='order_id', how='inner')
    .sort_values(['user_id','product_id','order_number'])
)

# 구매 간격 계산
pp_test['prev_cum_days'] = pp_test.groupby(['user_id','product_id'])['cum_days'].shift(1)
pp_test['purchase_interval'] = pp_test['cum_days'] - pp_test['prev_cum_days']

timing_agg = (
    pp_test.dropna(subset=['purchase_interval'])
    .groupby(['user_id','product_id'])['purchase_interval']
    .agg(up_avg_purchase_interval='mean',
         up_std_purchase_interval='std',
         up_interval_count='count')
    .reset_index()
)
timing_agg['up_std_purchase_interval'] = timing_agg['up_std_purchase_interval'].fillna(0)
timing_agg['up_purchase_regularity'] = (
    1 - timing_agg['up_std_purchase_interval'] / (timing_agg['up_avg_purchase_interval'] + 1e-6)
).clip(0, 1).astype(np.float32)

# fallback: 유저 평균 간격
user_fallback = (
    pp_test.dropna(subset=['purchase_interval'])
    .groupby('user_id')['purchase_interval'].mean()
    .reset_index().rename(columns={'purchase_interval': '_fb'})
)

candidates = (
    candidates
    .merge(timing_agg, on=['user_id','product_id'], how='left')
    .merge(user_fallback, on='user_id', how='left')
)
candidates['up_avg_purchase_interval'] = (
    candidates['up_avg_purchase_interval'].fillna(candidates['_fb'])
)
candidates.drop(columns=['_fb'], inplace=True)

# 파생 피처
candidates['days_until_expected'] = (
    candidates['up_avg_purchase_interval'] - candidates['days_since_prior_order']
).astype(np.float32)
candidates['timing_ratio'] = (
    candidates['days_since_prior_order'] / (candidates['up_avg_purchase_interval'] + 1e-6)
).astype(np.float32)
candidates['is_overdue'] = (candidates['timing_ratio'] >= 1.0).astype(np.int8)
candidates['up_interval_count'] = candidates['up_interval_count'].fillna(0).astype(np.int16)

print(f'구매 시점 피처 추가 완료  |  컬럼 수: {candidates.shape[1]}')

### 3-3. 공동구매 패턴 피처 생성

In [ ]:
# 공동구매 행렬 계산 (전체 prior 기준 — train+test 유저 동일하게 적용)
print('공동구매 행렬 계산 중...')

target_products = candidates['product_id'].unique()
pp_co = prior_products[prior_products['product_id'].isin(target_products)].copy()
pp_co = pp_co.merge(
    orders[orders['eval_set'] == 'prior'][['order_id','user_id']],
    on='order_id', how='inner'
)

basket_size = pp_co.groupby('order_id')['product_id'].transform('count')
pp_co = pp_co[basket_size.between(2, 30)].copy()

pid2idx = {p: i for i, p in enumerate(target_products)}
idx2pid = {i: p for p, i in pid2idx.items()}
order_ids_co = pp_co['order_id'].unique()
oid2idx = {o: i for i, o in enumerate(order_ids_co)}

mat = csr_matrix(
    (np.ones(len(pp_co), dtype=np.float32),
     (pp_co['order_id'].map(oid2idx).values,
      pp_co['product_id'].map(pid2idx).values)),
    shape=(len(order_ids_co), len(target_products))
)
print(f'sparse 행렬: {mat.shape[0]:,} × {mat.shape[1]:,}')

K_TOP = 10
BATCH = 200
n_products = len(target_products)
copurchase_records = []

print(f'top-{K_TOP} 파트너 추출 중...')
for start in range(0, n_products, BATCH):
    end  = min(start + BATCH, n_products)
    co   = mat[:, start:end].T.dot(mat).toarray()
    for li in range(end - start):
        gi = start + li
        co[li, gi] = 0
        top_idx = np.argsort(co[li])[::-1][:K_TOP]
        for rank, j in enumerate(top_idx):
            cnt = int(co[li, j])
            if cnt > 0:
                copurchase_records.append((idx2pid[gi], idx2pid[j], cnt, rank+1))
    if start % (BATCH * 20) == 0 and start > 0:
        print(f'  {end:,}/{n_products:,} 완료')

copurchase_pairs = pd.DataFrame(
    copurchase_records, columns=['product_id','partner_id','co_count','co_rank']
)
copurchase_pairs['co_score_norm'] = (
    copurchase_pairs['co_count'] /
    copurchase_pairs.groupby('product_id')['co_count'].transform('sum').clip(1)
).astype(np.float32)

print(f'\ncopurchase_pairs: {len(copurchase_pairs):,}행')

In [ ]:
# 유저 히스토리 (test 유저 기준)
user_hist = (
    pp_co[['user_id','product_id']]
    .drop_duplicates()
    .rename(columns={'product_id': 'partner_id'})
)
user_hist['in_history'] = np.int8(1)

# 친화도 점수 — 청크 방식
CHUNK = 500_000
total = len(candidates)
results = []

print(f'공동구매 친화도 계산 중... ({total:,}행)')
for start in range(0, total, CHUNK):
    chunk = candidates[['user_id','product_id']].iloc[start:start+CHUNK].copy()
    exp = chunk.merge(
        copurchase_pairs[['product_id','partner_id','co_count','co_score_norm']],
        on='product_id', how='left'
    )
    exp = exp.merge(user_hist, on=['user_id','partner_id'], how='left')
    exp['in_history']   = exp['in_history'].fillna(0)
    exp['weighted_hit'] = exp['co_score_norm'].fillna(0) * exp['in_history']

    agg = (
        exp.groupby(['user_id','product_id'], sort=False)
        .agg(copurchase_match_count   =('in_history',   'sum'),
             copurchase_partner_k     =('partner_id',   'count'),
             copurchase_weighted_score=('weighted_hit', 'sum'))
        .reset_index()
    )
    results.append(agg)
    print(f'  {min(start+CHUNK, total):,}/{total:,} 완료')

cp_final = pd.concat(results, ignore_index=True)
cp_final['copurchase_ratio'] = (
    cp_final['copurchase_match_count'] / cp_final['copurchase_partner_k'].clip(1)
).astype(np.float32)
cp_final['copurchase_weighted_score'] = cp_final['copurchase_weighted_score'].astype(np.float32)
cp_final['copurchase_match_count']    = cp_final['copurchase_match_count'].astype(np.int8)

candidates = candidates.merge(
    cp_final[['user_id','product_id','copurchase_ratio',
              'copurchase_weighted_score','copurchase_match_count']],
    on=['user_id','product_id'], how='left'
)
print(f'\n공동구매 피처 추가 완료  |  컬럼 수: {candidates.shape[1]}')

---
## 4. 피처 정렬 & 결측치 처리

In [ ]:
# 모델이 학습한 피처 순서에 맞게 정렬
# MODEL_FEATURES: 모델 학습 시 사용한 피처 목록

# 모델에 있지만 test에 없는 피처 확인
missing_in_test = [f for f in MODEL_FEATURES if f not in candidates.columns]
if missing_in_test:
    print(f'test에 없는 피처 ({len(missing_in_test)}개) → 0으로 채움:')
    for f in missing_in_test:
        print(f'  {f}')
        candidates[f] = 0
else:
    print('모든 모델 피처가 test에 존재합니다.')

# test에만 있는 불필요한 컬럼 제거 후 순서 정렬
X_test = candidates[list(MODEL_FEATURES)].copy()

# 결측치 처리 (중앙값 대체)
col_medians = df_train[list(MODEL_FEATURES)].median(numeric_only=True)
X_test = X_test.fillna(col_medians)

missing_after = X_test.isnull().sum().sum()
print(f'\n정렬 후 피처 수   : {X_test.shape[1]}')
print(f'총 후보 수        : {len(X_test):,}')
print(f'결측치 남은 수    : {missing_after}')

---
## 5. 예측 및 결과 생성

In [ ]:
print('재구매 확률 예측 중...')
prob = model.predict_proba(X_test)[:, 1]

candidates['reorder_prob'] = prob.astype(np.float32)
candidates['predicted']    = (prob >= best_thr).astype(np.int8)

print(f'\n=== 예측 결과 요약 ===')
print(f'전체 후보           : {len(candidates):,}')
print(f'재구매 예측 (1)     : {candidates["predicted"].sum():,}  ({candidates["predicted"].mean()*100:.2f}%)')
print(f'미구매 예측 (0)     : {(candidates["predicted"]==0).sum():,}')
print(f'주문당 평균 예측 수 : {candidates["predicted"].sum() / candidates["order_id"].nunique():.1f}개')

# 재구매 확률 분포
fig, ax = plt.subplots(figsize=(9, 3))
ax.hist(prob, bins=60, color='steelblue', alpha=0.7, edgecolor='white')
ax.axvline(best_thr, color='tomato', linewidth=2, label=f'threshold={best_thr:.2f}')
ax.set_xlabel('재구매 예측 확률')
ax.set_ylabel('후보 수')
ax.set_title('테스트 세트 재구매 확률 분포')
ax.legend()
plt.tight_layout()
plt.show()

---
## 6. order_products 형태로 저장

In [ ]:
# 재구매 예측된 것만 필터링
submission = (
    candidates[candidates['predicted'] == 1]
    [['order_id', 'product_id', 'reorder_prob']]
    .sort_values(['order_id', 'reorder_prob'], ascending=[True, False])
    .reset_index(drop=True)
)

# order_products__train.csv와 동일한 컬럼 구조로 저장
# add_to_cart_order: 확률 높은 순서대로 번호 부여
submission['add_to_cart_order'] = (
    submission.groupby('order_id').cumcount() + 1
).astype(np.int16)
submission['reordered'] = np.int8(1)  # 모두 재구매 예측

# 최종 컬럼 순서 (order_products__train.csv와 동일)
submission = submission[['order_id','product_id','add_to_cart_order','reordered','reorder_prob']]

print('=== 최종 예측 결과 ===')
print(f'예측된 주문 수    : {submission["order_id"].nunique():,} / 75,000')
print(f'예측된 상품 행 수 : {len(submission):,}')
print(f'주문당 평균 상품  : {len(submission)/submission["order_id"].nunique():.1f}개')
print(f'\n주문당 예측 상품 수 분포:')
per_order = submission.groupby('order_id').size()
print(per_order.describe().round(2))

submission.head(10)

In [ ]:
# 예측하지 못한 주문 확인 (재구매 예측이 0개인 주문 → 'None' 처리)
all_test_order_ids = set(test_orders['order_id'])
predicted_order_ids = set(submission['order_id'])
no_pred_orders = all_test_order_ids - predicted_order_ids

print(f'재구매 예측 있는 주문: {len(predicted_order_ids):,}')
print(f'재구매 예측 없는 주문: {len(no_pred_orders):,}  (→ None으로 처리)')

# 예측 없는 주문을 None 행으로 추가 (대회 제출 규격)
if no_pred_orders:
    none_rows = pd.DataFrame({
        'order_id'          : list(no_pred_orders),
        'product_id'        : 'None',
        'add_to_cart_order' : 0,
        'reordered'         : 0,
        'reorder_prob'      : 0.0
    })
    submission_full = pd.concat([submission, none_rows], ignore_index=True)
else:
    submission_full = submission.copy()

submission_full = submission_full.sort_values('order_id').reset_index(drop=True)

# 저장
out_path = PREP / 'submission_order_products.csv'
submission_full.to_csv(out_path, index=False, float_format='%.4f')

import os
size_mb = os.path.getsize(out_path) / 1024 / 1024
print(f'\n저장 완료: {out_path}')
print(f'행 수: {len(submission_full):,}  |  크기: {size_mb:.1f} MB')

In [ ]:
# 샘플 확인 — 특정 주문의 예측 결과
sample_order_id = submission['order_id'].iloc[0]

print(f'=== 주문 ID {sample_order_id} 예측 결과 ===')
sample = submission[submission['order_id'] == sample_order_id]
print(sample.to_string(index=False))

# 주문당 예측 상품 수 분포 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

per_order_cnt = submission.groupby('order_id').size()
axes[0].hist(per_order_cnt, bins=30, color='mediumseagreen', edgecolor='white')
axes[0].set_xlabel('주문당 예측 상품 수')
axes[0].set_ylabel('주문 수')
axes[0].set_title('주문당 예측 상품 수 분포')
axes[0].axvline(per_order_cnt.mean(), color='tomato', linestyle='--',
                label=f'평균 {per_order_cnt.mean():.1f}개')
axes[0].legend()

axes[1].hist(submission['reorder_prob'], bins=40, color='steelblue', edgecolor='white')
axes[1].set_xlabel('재구매 확률')
axes[1].set_ylabel('행 수')
axes[1].set_title('예측된 상품들의 재구매 확률 분포')

plt.tight_layout()
plt.show()

print(f'\n최종 파일: {out_path}')
print('완료!')